In [1]:

!nvidia-smi
!nvcc --version

Sun Oct 11 14:59:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile matmul_cuda.cu
#include <iostream>
#include <cstdlib>
#include <cuda_runtime.h>
#include <random>
#include <vector>
#include <cmath>
#include <algorithm>

using namespace std;

// check whether the cuda operation succeeded.
// we wrap all cuda operations in this to handle failures at any point
void checkCuda(cudaError_t result) {
    if (result != cudaSuccess) {
        cerr << cudaGetErrorString(result) << endl;
        exit(EXIT_FAILURE);
    }
}

__global__ void matmul(
    const float* A,
    const float* B,
    float* C,
    int M, int K, int N
) {
    // giving the thread its dimensions to execute
    int i = blockIdx.y * blockDim.y + threadIdx.y;
    int j = blockIdx.x * blockDim.x + threadIdx.x;


    if (i < M && j < N) {
        float sum = 0;

        // computing dot product
        for (int k = 0; k < K; k++) {
            sum += A[i*K + k] * B[k*N + j];
        }

        C[i*N + j] = sum;
    }
}

// basic matrix multiplication, used to compare to gpu matmul
void cpuMatmul(
    const float* A,
    const float* B,
    float* C,
    int M, int K, int N
) {
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;

            for (int k = 0; k < K; k++) {
                sum += A[i*K + k] * B[k*N + j];
            }

            C[i*N + j] = sum;
        }
    }
}

int main() {
    // Using example matrices; they can be arbitrarily large
    const int M = 55;
    const int K = 66;
    const int N = 77;

    unsigned int seed = 100;
    mt19937 gen(seed);
    uniform_real_distribution<float> distrib(-1.0f, 1.0f);

    vector<float> h_A(M*K);
    vector<float> h_B(K*N);
    vector<float> h_C(M*N);

    for (int i = 0; i < M; i++){
      for (int j = 0; j < K; j++){
        h_A[i * K + j] = distrib(gen);
      }
    }

    for (int i = 0; i < K; i++){
      for (int j = 0; j < N; j++){
        h_B[i * N + j] = distrib(gen);
      }
    }


    float* d_A = nullptr;
    float* d_B = nullptr;
    float* d_C = nullptr;

    size_t bytesA = M * K * sizeof(float);
    size_t bytesB = K * N * sizeof(float);
    size_t bytesC = M * N * sizeof(float);

    checkCuda(cudaMalloc((void**)&d_A, bytesA));
    checkCuda(cudaMalloc((void**)&d_B, bytesB));
    checkCuda(cudaMalloc((void**)&d_C, bytesC));

    checkCuda(cudaMemcpy(d_A, h_A.data(), bytesA, cudaMemcpyHostToDevice));
    checkCuda(cudaMemcpy(d_B, h_B.data(), bytesB, cudaMemcpyHostToDevice));

    // initializing block of threads
    // most efficient size depends on the size of input matrix
    dim3 block(16, 16);

    dim3 grid(
        (N + block.x - 1) / block.x,
        (M + block.y - 1) / block.y
    );


    // executes on the GPU
    matmul<<<grid, block>>>(d_A, d_B, d_C, M, K, N);

    checkCuda(cudaGetLastError());
    checkCuda(cudaDeviceSynchronize());

    // copy result from GPU back to CPU
    checkCuda(cudaMemcpy(h_C.data(), d_C, bytesC, cudaMemcpyDeviceToHost));

    // create and compute an array to store the CPU reference result
    std::vector<float> expected(M * N);
    cpuMatmul(h_A.data(), h_B.data(), expected.data(), M, K, N);

    // now, calculate the error in our GPU-calculated matrix
    const float atol = 1e-3f;
    const float rtol = 1e-4f;

    float maxError = 0.0f;
    int failures = 0;

    for (int i = 0; i < M * N; i++) {
        float gpuValue = h_C[i];
        float cpuValue = expected[i];

        float error = fabs(gpuValue - cpuValue);

        float tolerance = atol + rtol * fabs(cpuValue);

        // track the largest absolute error
        if (std::isfinite(error)) {
            maxError = max(maxError, error);
        }

        // check for invalid numbers or excessive error
        if (!std::isfinite(gpuValue) ||
            !std::isfinite(cpuValue) ||
            error > tolerance) {

            failures++;

            // Print only the first few failures.
            if (failures <= 5) {
                std::cout << "Mismatch at ("
                          << i / N << ", " << i % N << "): "
                          << "CPU = " << cpuValue
                          << ", GPU = " << gpuValue
                          << std::endl;
            }
        }
    }

    std::cout << "Maximum absolute error: "
              << maxError << std::endl;

    if (failures == 0) {
        std::cout << "PASS: All values match within tolerance."
                  << std::endl;
    } else {
        std::cout << "FAIL: " << failures
                  << " values did not match."
                  << std::endl;
    }

    checkCuda(cudaFree(d_A));
    checkCuda(cudaFree(d_B));
    checkCuda(cudaFree(d_C));

    return failures == 0 ? 0: 1;
}


Writing matmul_cuda.cu


In [5]:
!nvcc matmul_cuda.cu -o matmul_cuda
!./matmul_cuda

Maximum absolute error: 2.38419e-06
PASS: All values match within tolerance.


In [ ]:

import os
import subprocess
from google.colab import userdata

repo = "/content/cuda-matmul-optimization"

# Retrieve your existing token from Colab Secrets.
token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("GITHUB_TOKEN is missing.")

# Set up authentication for these commands.
env = os.environ.copy()
env["GH_TOKEN"] = token

# Configure Git to use GitHub CLI.
subprocess.run(
    ["gh", "auth", "setup-git",
     "--hostname", "github.com", "--force"],
    cwd=repo,
    env=env,
    check=True
)

# Push your already-created commit.
result = subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("SUCCESS! Your latest commit is on GitHub.")
else:
    print("Push failed:")
    print(result.stderr.replace(token, "[REDACTED]"))


In [ ]:
import torch
import numpy as np
import time
import statistics
from pathlib import Path

root = Path("weights/mlp")

# load the same inputs and parameters as the CUDA program
X = torch.from_numpy(
    np.fromfile(root / "full_test/inputs_cuda.bin",
                dtype=np.float32).reshape(10000, 784)
)

W1 = torch.from_numpy(
    np.fromfile(root / "weights1_cuda.bin",
                dtype=np.float32).reshape(784, 128)
)

b1 = torch.from_numpy(
    np.fromfile(root / "biases1_cuda.bin", dtype=np.float32)
)

W2 = torch.from_numpy(
    np.fromfile(root / "weights2_cuda.bin",
                dtype=np.float32).reshape(128, 10)
)

b2 = torch.from_numpy(
    np.fromfile(root / "biases2_cuda.bin", dtype=np.float32)
)

# define inference using PyTorch's CPU operations
def inference_cpu():
    H = torch.relu(X @ W1 + b1)
    Z = H @ W2 + b2
    return Z

warmup_runs = 5
measured_runs = 30
times = []

with torch.inference_mode():

    # verify correctness first
    result = inference_cpu()

    reference = torch.from_numpy(
        np.fromfile(
            root / "full_test/reference_logits.bin",
            dtype=np.float32
        ).reshape(10000, 10)
    )

    print("Matches PyTorch reference:",
          torch.allclose(result, reference,
                         atol=1e-3, rtol=1e-4))

    # Warm-up runs
    for _ in range(warmup_runs):
        inference_cpu()

    # Measured runs
    for _ in range(measured_runs):
        start = time.perf_counter()

        inference_cpu()

        elapsed_ms = (time.perf_counter() - start) * 1000
        times.append(elapsed_ms)

print("CPU threads:", torch.get_num_threads())
print(f"Minimum CPU time: {min(times):.4f} ms")
print(f"Median CPU time: {statistics.median(times):.4f} ms")
print(f"Maximum CPU time: {max(times):.4f} ms")


Matches PyTorch reference: True
CPU threads: 1
Minimum CPU time: 19.0675 ms
Median CPU time: 21.0740 ms
Maximum CPU time: 30.3605 ms


In [ ]:
import torch
import statistics

# verify GPU availability
assert torch.cuda.is_available(), "CUDA GPU unavailable"

device = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))

# transfer inputs and weights to the GPU once,
# before starting measurements.
X_gpu = X.to(device)
W1_gpu = W1.to(device)
b1_gpu = b1.to(device)
W2_gpu = W2.to(device)
b2_gpu = b2.to(device)

# define the same MLP forward pass
def inference_gpu():
    H = torch.relu(X_gpu @ W1_gpu + b1_gpu)
    Z = H @ W2_gpu + b2_gpu
    return Z

warmup_runs = 5
measured_runs = 30
times = []

with torch.inference_mode():

    # check correctness against the CPU reference
    result = inference_gpu()

    print(
        "Matches PyTorch reference:",
        torch.allclose(
            result.cpu(), reference,
            atol=1e-3, rtol=1e-4
        )
    )

    # Warm up the GPU
    for _ in range(warmup_runs):
        inference_gpu()

    torch.cuda.synchronize()

    # measure inference with CUDA events
    for _ in range(measured_runs):

        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)

        start.record()

        inference_gpu()

        stop.record()
        stop.synchronize()

        elapsed_ms = start.elapsed_time(stop)
        times.append(elapsed_ms)

print(f"Minimum PyTorch GPU time: {min(times):.4f} ms")
print(f"Median PyTorch GPU time: {statistics.median(times):.4f} ms")
print(f"Maximum PyTorch GPU time: {max(times):.4f} ms")


GPU: Tesla T4
Matches PyTorch reference: True
Minimum PyTorch GPU time: 0.6144 ms
Median PyTorch GPU time: 0.6185 ms
Maximum PyTorch GPU time: 0.6554 ms


In [2]:
%cd /content
!git clone https://github.com/loganKneller/cuda-matmul-optimization.git
%cd /content/cuda-matmul-optimization
!git log --oneline -5


/content
Cloning into 'cuda-matmul-optimization'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 80 (delta 27), reused 61 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 26.47 MiB | 7.34 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/cuda-matmul-optimization
31f3612 (HEAD -> main, origin/main, origin/HEAD) Added average-time testing
0d5dfe0 Added benchmark testing for 10000 images
a5a2b5f Added inference for full MNIST data set
4849cd1 Removed leftover referenceFile code
baaeca7 Implemented MLP model prediction in CUDA


In [3]:
%cd /content/cuda-matmul-optimization
!git pull
!make mlp_benchmark ARCH=sm_75
!./mlp_benchmark


/content/cuda-matmul-optimization
Already up to date.
Successfully loaded all MLP data
Successfully loaded PyTorch reference
Successfully loaded 10000 MNIST labels
Successfully copied MLP data to GPU!
Warm-up runs: 5
Measured runs: 30
Minimum CUDA inference time: 5.8036 ms
Median CUDA inference time: 5.8081 ms
Maximum CUDA inference time: 7.0636 ms
Amortized median time per image: 0.0006 ms
MLP forward pass completed and output copied to CPU
Image 0: CUDA predicts 7, PyTorch predicts 7
Image 1: CUDA predicts 2, PyTorch predicts 2
Image 2: CUDA predicts 1, PyTorch predicts 1
Image 3: CUDA predicts 0, PyTorch predicts 0
Image 4: CUDA predicts 4, PyTorch predicts 4
Image 5: CUDA predicts 1, PyTorch predicts 1
Image 6: CUDA predicts 4, PyTorch predicts 4
Image 7: CUDA predicts 9, PyTorch predicts 9
Image 8: CUDA predicts 5, PyTorch predicts 5
Image 9: CUDA predicts 9, PyTorch predicts 9
Maximum PyTorch error: 9.5367e-06
Logit mismatches: 0
Matching predictions: 10000/10000
Correct predicti